# marv-hyena round 5: controls that could explain round 4 away

A review of round 4 (see `RESEARCH_LOG.md`, 2026-09-25) found that its two most novel claims had not faced the
control that could undo them:

- **"Evo 2 knows amino acids" (P21, 68.9%)**: at the third codon letter the *silent* change is almost always a
  **transition** (A↔G, C↔T) and the *missense* change a **transversion**. On the 17 sites where both were
  transversions, missense won 8/17. Round 5 holds letter type fixed, flips it, and removes the protein entirely.
- **"SE blocks apply the genetic code" (90.8%)**: chosen by the largest missense/silent *ratio* per block, which
  favours blocks where the silent number is tiny. Round 5 keeps every block's numbers and ranks by *difference*.
- **"Block 0 is learned: 46 vs 0"**: one hard cutoff (80%). Round 5 looks at the whole distribution.
- **Stops (P22)**: gated on the wrong arm and underpowered. Round 5 uses the whole genome and paired designs.

| step | question | prediction |
|---|---|---|
| R5.1 | Letter type alone, no protein change (four-fold sites, and DNA outside genes) | P23, P24 |
| R5.2 | Amino acids with letter type held fixed, and with it turned *against* the hypothesis | P25 |
| R5.3 | SE-peak recheck: every block kept, ranked three ways, against a no-amino-acid control | P26 |
| R5.4 | Premature stops vs missense, matched and flipped | P27 |
| R5.5 | Block-0 null without the cutoff | P28 |

Predictions P23–P28 are registered in `PREDICTIONS.md` **before** this runs. Do not edit them afterwards.

**Runtime:** A100 + High-RAM. About 45–75 minutes of GPU time at the default site counts.

## 0. Setup

In [ ]:
# GPU check (no torch import yet: the install below may change the torch version)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# OPTIONAL: keep the ~15 GB Evo 2 weights on Google Drive so the next session skips the download.
# This must run BEFORE anything imports huggingface_hub / evo2.
USE_DRIVE = False

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF_HOME =', os.environ.get('HF_HOME', '(default ~/.cache/huggingface)'))

In [ ]:
# Install Evo 2 + marv-hyena (~2-5 min). NO flash-attn: pip usually finds no prebuilt wheel for Colab's torch
# and compiles for hours. marv-hyena instead runs Evo 2's attention through PyTorch's built-in fused kernel
# (scaled_dot_product_attention), which is also fast on an A100 -- see marv_hyena/noflash.py.
# Colab runs Python 3.13, but every current evo2 release declares Python < 3.13, so a plain `pip install evo2`
# silently falls back to the old, incompatible 0.3.0. evo2 is pure Python, so force the current release.
# Colab also ships an empty `transformer-engine` package whose import crashes evo2; remove it (7B needs no TE).
!pip uninstall -y -q transformer-engine transformer_engine
!pip install -q --ignore-requires-python evo2==0.6.0
BRANCH = 'main'
!rm -rf marv-hyena && git clone -q -b {BRANCH} https://github.com/thebnbrkr/marv-hyena.git
!pip install -q -r marv-hyena/requirements.txt openpyxl

import sys
sys.path.insert(0, '/content/marv-hyena')

# Fail loudly now rather than 20 minutes in, after the 15 GB model download.
import subprocess, os
print('branch:', subprocess.run(['git', '-C', 'marv-hyena', 'rev-parse', '--abbrev-ref', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())
for m in ('nullmodel', 'genome', 'codons', 'controls'):
    assert os.path.exists(f'marv-hyena/marv_hyena/{m}.py'), (
        f'{m}.py missing: wrong branch. round 5 needs a checkout that includes controls.py')
print('round-5 modules present')

In [ ]:
# Must run before anything imports evo2/vortex: lets Vortex import without flash-attn and turns flash attention off.
from marv_hyena import noflash
noflash.prepare()

import torch, vortex, evo2
from importlib.metadata import version
print('evo2', version('evo2'), '| vtx', version('vtx'))
assert version('evo2') >= '0.6.0', 'old evo2 installed: Runtime -> Disconnect and delete runtime, then rerun from the top'
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| flash-attn package installed:', noflash.flash_attn_available())
print('GPU', torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0)[0] >= 8, 'needs an Ampere+ GPU with bf16 (A100 / L4 / H100)'

In [ ]:
# marv-hyena's own tests (a tiny CPU model that mirrors Vortex). Should say "83 passed".
!cd marv-hyena && python -m pytest -q

In [ ]:
# Data: the annotated E. coli K-12 genome from the evo2 repo
import os, urllib.request
os.makedirs('data', exist_ok=True)
EVO2_RAW = 'https://raw.githubusercontent.com/ArcInstitute/evo2/main/notebooks'
GENOME = 'data/NC_000913.gb'
if not os.path.exists(GENOME):
    urllib.request.urlretrieve(f'{EVO2_RAW}/sparse_autoencoder/NC_000913.gb', GENOME)

import marv_hyena as mh
from marv_hyena.probes import load_sequence
genome = load_sequence(GENOME)
print(f'E. coli genome: {len(genome):,} letters')

In [ ]:
# Load Evo 2 7B (downloads ~15 GB the first time)
import time
MODEL_NAME = 'evo2_7b'
t0 = time.time()
hm = mh.HyenaModel.load(MODEL_NAME)
print(f'loaded in {time.time()-t0:.0f}s')
print(hm.describe())
RESULTS = {}

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, random
from marv_hyena import diagnostics, motifs, nullmodel, codons, controls
from marv_hyena.checks import run_smoke_checks

seq = genome[100_000:104_096]
assert run_smoke_checks(hm, seq), 'smoke checks failed: stop here'
HEALTH_SEQ = genome[300_000:304_096]
BOTTLENECK = diagnostics.find_bottlenecks(hm, seq, [1000, 2000, 4095])
print('bottleneck:', BOTTLENECK, '| baseline health:', diagnostics.health(hm, HEALTH_SEQ))
L = BOTTLENECK[0] if BOTTLENECK else hm.n_blocks - 2   # evo2_7b: block 30
print('interface block:', L)

# Round 3's load-bearing set, re-found on this run: family ablations must keep these on.
lb = diagnostics.find_load_bearing(hm, HEALTH_SEQ)
LOAD_BEARING = sorted({r['block'] for r in lb if r['broken']})
print('load-bearing mixers:', LOAD_BEARING)

RESULTS = {'bottleneck': BOTTLENECK, 'interface_block': L, 'load_bearing': lb,
           'load_bearing_blocks': LOAD_BEARING}

# One reference we come back to: any experiment that scrambles weights must put
# them back, and this is what "put them back" is checked against.
REF_LOGITS = hm.logits(HEALTH_SEQ).clone()

## R5.0 Sites from the whole genome

Round 4 used a 60,000-letter window, which gave 119 usable sites and only 11 stops. Round 5 annotates the whole
E. coli genome and draws sites spread across all of it. Every design picks its two substitutions **at the same
position**, by (kind, letter type):

| design | arm a | arm b | what it isolates |
|---|---|---|---|
| `round4` | silent (mostly transition) | missense (mostly transversion) | round 4's exact sites, rerun with every block kept |
| `fourfold` | silent transition | silent transversion | letter type alone, protein unchanged |
| `noncoding` | transition | transversion | letter type alone, no gene at all |
| `matched` | silent transversion | missense transversion | amino acid, letter type held fixed |
| `flipped` | silent transversion | missense transition | amino acid, letter type pushing the *other* way |
| `stop_matched` | stop, transversion | missense, transversion | stop vs amino-acid swap, letter type fixed |
| `stop_flipped` | stop, transition | missense, transversion | stop vs swap, letter type against the stop |

In [ ]:
import time
N_SITES = 150        # per design; lower for a quick run
rng_sites = random.Random(0)

whole = mh.genbank_track(GENOME, 0, len(genome))
usage = codons.codon_usage(whole)
print(f'whole-genome track: {len(whole.seq):,} letters | forward-strand codons counted for usage: '
      f'{sum(1 for p in whole.codon_phase if p == 0):,}')

def spread(sites, n):
    '''Draw n sites spread over the whole genome instead of the first n.'''
    return sorted(rng_sites.sample(sites, n), key=lambda s: s.pos) if len(sites) > n else sites

all_designs = codons.design_sites(whole, max_sites=None, min_spacing=300, seed=0)
DESIGNS = {k: spread(v, N_SITES) for k, v in all_designs.items()}

# round 4's exact sites, so the SE-peak claim is rechecked on the same data
track4 = mh.genbank_track(GENOME, 200_000, 260_000)
r4_sites = codons.from_wobble(codons.wobble_sites(track4, allow_stop=False, min_spacing=120, max_sites=120))
for s in r4_sites:
    s.pos += track4.start   # window index -> genome index
DESIGNS = {'round4': r4_sites, **DESIGNS}

print(pd.DataFrame([{'design': k, 'available': len(all_designs.get(k, v)), 'used': len(v),
                     'a': v[0].a.label if v else '', 'b': v[0].b.label if v else '',
                     'amino acids': ''.join(sorted({s.aa for s in v if s.aa}))}
                    for k, v in DESIGNS.items()]).to_string(index=False))
RESULTS['design_counts'] = {k: len(v) for k, v in DESIGNS.items()}

In [ ]:
# Run every design. Each site: both arms' downstream effect + every block's divergence.
PAIRED = {}
for name, sites in DESIGNS.items():
    t0 = time.time()
    PAIRED[name] = codons.paired_test(hm, genome, sites, window=4096, span=24, downstream_span=200, usage=usage)
    print(f'{name:13s} {len(PAIRED[name]):4d} sites  {time.time() - t0:6.0f}s')
RESULTS['paired'] = PAIRED
SUMMARY = {k: controls.summarize_paired(v) for k, v in PAIRED.items()}
RESULTS['paired_summary'] = SUMMARY
print()
print(pd.DataFrame(SUMMARY.values())[['design', 'a', 'b', 'n', 'b_more_disruptive_frac', 'ci95', 'sign_test_p',
                                      'median_effect_a', 'median_effect_b']].to_string(index=False))

## R5.1 Letter type alone (P23, P24)

In `fourfold` and `noncoding` **no amino acid changes**. If transversions still disturb the model more there — at
about the rate round 4 credited to the genetic code — then round 4's 68.9% is explained by letter type.

In [ ]:
ff, nc = SUMMARY['fourfold'], SUMMARY['noncoding']
print(f"four-fold sites: transversion more disruptive than transition, same site : {ff['b_more_disruptive_frac']:.1%} "
      f"(n={ff['n']}, 95% CI {ff['ci95'][0]:.2f}-{ff['ci95'][1]:.2f})   P23 predicted >= 60%")
print(f"outside genes  : transversion more disruptive than transition, same site : {nc['b_more_disruptive_frac']:.1%} "
      f"(n={nc['n']}, 95% CI {nc['ci95'][0]:.2f}-{nc['ci95'][1]:.2f})   P24 predicted >= 55%, and below four-fold")

## R5.2 Amino acids, with letter type controlled (P25)

`matched`: both arms are transversions. `flipped`: the silent arm is the transversion, so letter type now works
**against** "missense disturbs more". Then one regression over every paired design: the within-site difference
against each candidate explanation at once (amino acid changed, protein ended, transversion, G/C change, codon
rarity). Both arms share the site, so everything about the site cancels.

In [ ]:
m, f = SUMMARY['matched'], SUMMARY['flipped']
print(f"matched (both transversions): missense more disruptive {m['b_more_disruptive_frac']:.1%} "
      f"(n={m['n']}, CI {m['ci95'][0]:.2f}-{m['ci95'][1]:.2f}, p={m['sign_test_p']:.3g})   P25 predicted >= 60%")
print(f"flipped (silent is the transversion): missense more disruptive {f['b_more_disruptive_frac']:.1%} "
      f"(n={f['n']}, CI {f['ci95'][0]:.2f}-{f['ci95'][1]:.2f})   P25 predicted >= 50%")
print('\nby amino acid (matched):')
print(pd.DataFrame(controls.stratify(PAIRED['matched'], lambda r: r['aa'] + ' ' + r['codon'][:2] + '.')).T
      [['n', 'b_more_disruptive_frac']].to_string())

pooled = [r for k in ('round4', 'fourfold', 'noncoding', 'matched', 'flipped') for r in PAIRED[k]]
fit = controls.paired_regression(pooled)
RESULTS['paired_regression'] = fit
print(f"\npaired regression over {fit['n']} sites (negative = makes the change MORE disruptive, nats over 200 letters):")
for k, v in fit['coef'].items():
    lo, hi = fit['ci95'][k]
    print(f"  {k:15s} {v:+8.3f}   95% CI {lo:+.3f} .. {hi:+.3f}")
print('  not estimable (never varied):', fit['dropped'])
print('\nP25 decides on: d_missense CI entirely below 0, with d_transversion in the model.')

## R5.3 The SE-peak claim, rechecked (P26)

Three ways to pick "the block where the two arms separate most", on round 4's **own sites**, then the same on the
four-fold design where no amino acid changes. If SE still wins under `diff`, and does **not** win when no amino acid
changes, the claim survives.

In [ ]:
# Architecture null: the same round-4 sites through Evo 2 with EVERY block's weights shuffled.
# On marv-hyena's untrained tiny test model this statistic already put 64% of peaks in SE blocks
# (base rate 25%), so the SE preference may be a property of short filters, not of the genetic code.
N_NULL = 60
with nullmodel.random_weights(hm, range(hm.n_blocks), mode='shuffle', seed=0):
    PAIRED['round4_shuffled'] = codons.paired_test(hm, genome, DESIGNS['round4'][:N_NULL], window=4096, span=24,
                                                   downstream_span=200, usage=usage)
chk = nullmodel.verify_restored(hm, HEALTH_SEQ, REF_LOGITS)
assert chk['restored'], 'weights were not restored; every later number is invalid'
RESULTS['paired']['round4_shuffled'] = PAIRED['round4_shuffled']

rows = []
for name in ('round4', 'round4_shuffled', 'fourfold', 'noncoding', 'matched'):
    for method in ('ratio', 'ratio_floor', 'diff'):
        s = controls.peak_kind_shares(PAIRED[name], method=method)
        rows.append({'design': name, 'method': method, 'n': s['n'], 'modal block': s['modal_block'],
                     **{f'{k} share': round(v, 3) for k, v in s['share'].items()}})
peaks = pd.DataFrame(rows)
RESULTS['peak_shares'] = rows
print('base rate of each kind among 32 blocks:', {k: round(v, 3) for k, v in
      controls.peak_kind_shares(PAIRED['round4'])['base_rate'].items()})
print(peaks.to_string(index=False))

# the per-block picture behind it: median divergence of each arm, round-4 sites
kinds = PAIRED['round4'][0]['kinds']
da = np.median([r['div_a'] for r in PAIRED['round4']], axis=0)
db = np.median([r['div_b'] for r in PAIRED['round4']], axis=0)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].plot(da, label='silent'); ax[0].plot(db, label='missense'); ax[0].set_yscale('log'); ax[0].legend()
ax[0].set_title('median relative divergence per block (round-4 sites)'); ax[0].set_xlabel('block')
ax[1].bar(range(len(kinds)), db - da, color=['C0' if k == 'se' else 'C7' for k in kinds])
ax[1].set_title('missense minus silent (blue = SE block)'); ax[1].set_xlabel('block')
plt.tight_layout(); plt.show()

## R5.4 Premature stops, done properly (P27)

Paired at the same site, whole genome, gated on nothing: the sign test uses every site. `stop_flipped` makes the stop
a transition and the missense a transversion, so letter type argues against the stop.

In [ ]:
for k in ('stop_matched', 'stop_flipped'):
    s = SUMMARY[k]
    print(f"{k:13s}: stop more disruptive than missense at the same site {s['b_more_disruptive_frac']:.1%} "
          f"-> reported as 1 - that = {1 - s['b_more_disruptive_frac']:.1%}  (n={s['n']})")
    print(f"               median effect: stop {s['median_effect_a']:+.2f}  missense {s['median_effect_b']:+.2f} nats")
print('\nP27 predicted: stop more disruptive in >= 90% (matched) and >= 85% (flipped); untestable if n < 30.')
print('Note: arm a is the stop here, so "stop more disruptive" = 1 - b_more_disruptive_frac.')
s = controls.peak_kind_shares(PAIRED['stop_matched'], method='diff')
print('where stop and missense separate most (diff):', {k: round(v, 2) for k, v in s['share'].items()},
      '| modal block', s['modal_block'])

## R5.5 Block 0 without the cutoff (P28)

For each live channel: the largest share of its top-50 inputs that contain any one three-letter word. Trained vs
weight-shuffled (3 seeds) vs Gaussian (3 seeds), then how many channels clear each cutoff from 0.5 to 0.9.

In [ ]:
md_t = motifs.enumerate_block0(hm)
live_t = controls.live_channels(md_t)
share = {'trained': controls.best_word_share(md_t, live=live_t)}
for mode in ('shuffle', 'gaussian'):
    for sd in (0, 1, 2):
        with nullmodel.random_weights(hm, 0, mode=mode, seed=sd):
            md_n = motifs.enumerate_block0(hm)
        share[f'{mode}{sd}'] = controls.best_word_share(md_n, live=controls.live_channels(md_n))
chk = nullmodel.verify_restored(hm, HEALTH_SEQ, REF_LOGITS)
assert chk['restored'], 'block 0 was not restored; every later number is invalid'

sweep = {k: controls.cutoff_sweep(v) for k, v in share.items()}
RESULTS['block0_share'] = {k: {'n_live': int(len(v)), 'median': float(np.median(v)),
                               'q90': float(np.quantile(v, .9)), 'sweep': sweep[k]} for k, v in share.items()}
print(pd.DataFrame(RESULTS['block0_share']).T.to_string())
fig, ax = plt.subplots(figsize=(8, 3.5))
for k, v in share.items():
    ax.hist(v, bins=np.linspace(0, 1, 41), histtype='step', lw=2 if k == 'trained' else 1, label=k)
ax.axvline(0.8, color='k', ls=':', lw=1); ax.set_xlabel("best one-word share of a channel's top-50 inputs")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
t_med = np.median(share['trained']); n_med = np.median([np.median(share[f'shuffle{s}']) for s in range(3)])
print(f'trained median {t_med:.3f} vs shuffled median {n_med:.3f}   P28 predicted a gap >= 0.20')
print('P28 also predicted trained > every null at every cutoff 0.5-0.9:',
      all(sweep['trained'][c] > sweep[k][c] for k in sweep if k != 'trained' for c in sweep['trained']))

## Save

`results_round5.json` goes next to the executed notebook in `results/round5/`. Record the outcomes in
`PREDICTIONS.md` **as they came out**, then write the round-5 entry in `RESEARCH_LOG.md`.

In [ ]:
import json

def jsonable(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.ndarray,)): return o.tolist()
    if isinstance(o, (np.bool_,)): return bool(o)
    return str(o)

with open('results_round5.json', 'w') as fh:
    json.dump(RESULTS, fh, default=jsonable, indent=1)
print('wrote results_round5.json', os.path.getsize('results_round5.json'), 'bytes')
print('keys:', list(RESULTS))

try:
    from google.colab import files
    files.download('results_round5.json')
except Exception as e:
    print('(download skipped:', e, ')')